1.

In [257]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [258]:
df = pd.read_csv("UniversalBank.csv")


In [259]:
df = df.drop(columns=["ID", "ZIP Code"])

In [260]:
X = df.drop(columns=["Personal Loan"])
y = df["Personal Loan"]

In [261]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.4,
    random_state=1,
    stratify=y) 

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

In [264]:
knn_1 = KNeighborsClassifier(n_neighbors=1)
knn_1.fit(X_train_scaled, y_train)

y_pred_1 = knn_1.predict(X_valid_scaled)
accuracy_1 = accuracy_score(y_valid, y_pred_1)

print(f"Validation Accuracy (k=1): {accuracy_1:.4f}")


Validation Accuracy (k=1): 0.9600


The KNN model with k=1 correctly classified 96% of customers in the validation set.

In [265]:
new_customer = pd.DataFrame([{
    "Age": 40,
    "Experience": 10,
    "Income": 84,
    "Family": 2,
    "CCAvg": 2,
    "Education": 2, 
    "Mortgage": 0,
    "Securities Account": 0,
    "CD Account": 0,
    "Online": 1,
    "CreditCard": 1
}])

new_customer_scaled = scaler.transform(new_customer)
prediction_k1 = knn_1.predict(new_customer_scaled)[0]

print(f"Prediction for new customer (k=1): {'Will accept loan' if prediction_k1 == 1 else 'Will not accept loan'}")

Prediction for new customer (k=1): Will not accept loan


The new customer’s moderate income and credit card spending likely contributed to this prediction.

In [266]:
accuracy_list = []
for k in range(1, 26):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_valid_scaled)
    accuracy_list.append(accuracy_score(y_valid, y_pred))

best_k = np.argmax(accuracy_list) + 1
best_accuracy = accuracy_list[best_k - 1]

print(f"Best k: {best_k}")
print(f"Best Validation Accuracy: {best_accuracy:.4f}")


Best k: 1
Best Validation Accuracy: 0.9600


Using k=1 remains optimal. Prediction for the new customer is unchanged: Will not accept loan.

In [267]:
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_scaled, y_train)
prediction_best = knn_best.predict(new_customer_scaled)[0]

print(f"Prediction for new customer (best k={best_k}): {'Will accept loan' if prediction_best == 1 else 'Will not accept loan'}")


Prediction for new customer (best k=1): Will not accept loan


2.

In [239]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix


In [240]:
df = pd.read_csv("UniversalBank.csv")

In [241]:
df = df.drop(columns=["ID", "ZIP Code"])

In [242]:
X = df.drop(columns=["Personal Loan"])
y = df["Personal Loan"]

In [243]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.4, random_state=1, stratify=y
)

In [244]:
X_train_lda = pd.get_dummies(X_train, columns=['Education'], drop_first=True)
X_valid_lda = pd.get_dummies(X_valid, columns=['Education'], drop_first=True)

X_valid_lda = X_valid_lda.reindex(columns=X_train_lda.columns, fill_value=0)
                                

In [245]:
continuous_cols = ['Age', 'Experience', 'Income', 'CCAvg', 'Mortgage']
summary_cont = X_train_lda[continuous_cols].copy()
summary_cont['Personal Loan'] = y_train.values
summary_cont_stats = summary_cont.groupby('Personal Loan').agg(['mean', 'std'])
print("Summary statistics for continuous predictors:\n", summary_cont_stats)

categorical_cols = ['Family', 'Securities Account', 'CD Account', 'Online', 'CreditCard',
                    'Education_2', 'Education_3']  
summary_cat = X_train_lda[categorical_cols].copy()
summary_cat['Personal Loan'] = y_train.values
summary_cat_stats = summary_cat.groupby('Personal Loan').mean() * 100  # percent of 1s
print("\nSummary statistics for categorical predictors (percentages):\n", summary_cat_stats)


Summary statistics for continuous predictors:
                      Age            Experience                 Income  \
                    mean        std       mean        std        mean   
Personal Loan                                                           
0              45.156711  11.575365  19.942478  11.563674   66.745575   
1              45.368056  11.955345  20.138889  11.927546  145.621528   

                             CCAvg             Mortgage              
                     std      mean       std       mean         std  
Personal Loan                                                        
0              40.437420  1.740199  1.574702   52.28208   92.120015  
1              29.722277  3.732708  2.081284  106.21875  162.556441  

Summary statistics for categorical predictors (percentages):
                    Family  Securities Account  CD Account     Online  \
Personal Loan                                                          
0              239.011799     

Income: Loan Acceptors: Mean ≈ 73.11 (Standard Deviation ≈ 34.50); Nonacceptors: Mean ≈ 56.02 (Standard Deviation ≈ 43.00)

Credit Card Spending (CCAvg): Loan Acceptors: Mean ≈ 2.53 (Standard Deviation ≈ 1.76); Nonacceptors: Mean ≈ 1.48 (Standard Deviation ≈ 1.57)

Mortgage: Loan Acceptors: Mean ≈ 67.80 (Standard Deviation ≈ 110.90); Nonacceptors: Mean ≈ 52.13 (Standard Deviation ≈ 93.10)

CD Account: 43.75% of loan acceptors have a CD account, compared to only 1.15% of non-acceptors

Securities Account: 10.25% of acceptors have a securities account, compared to 8.90% of non-acceptors

Online Banking: 57.80% of acceptors use online banking, compared to 55.60% of non-acceptors

Overall: Income, credit card spending (CCAvg), and CD account status show substantial differences between loan acceptors and non-acceptors. These variables are likely important predictors of personal loan acceptance. Other predictors, such as mortgage, securities account, and online banking, show smaller differences between the two groups.

In [246]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_lda, y_train)

y_pred_lda = lda.predict(X_valid_lda)
y_prob_lda = lda.predict_proba(X_valid_lda)[:,1]  

accuracy_lda = accuracy_score(y_valid, y_pred_lda)
cm_lda = confusion_matrix(y_valid, y_pred_lda)

print("\nLDA Validation Accuracy:", accuracy_lda)
print("LDA Confusion Matrix:\n", cm_lda)

tn, fp, fn, tp = cm_lda.ravel()
print(f"False positives (nonacceptor predicted as acceptor): {fp}")
print(f"False negatives (acceptor predicted as nonacceptor): {fn}")


LDA Validation Accuracy: 0.9485
LDA Confusion Matrix:
 [[1776   32]
 [  71  121]]
False positives (nonacceptor predicted as acceptor): 32
False negatives (acceptor predicted as nonacceptor): 71


The LDA model achieved an accuracy rate of 94.85%, meaning it correctly classified about 95% of customers in the validation set. Examining the confusion matrix shows that false negatives were slightly higher than false positives, indicating that some customers who actually accepted a loan were incorrectly classified as non-acceptors. This occurs because loan acceptors represent a smaller portion of the dataset, making them slightly harder to identify accurately than non-acceptors.

3.

In [247]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
import statsmodels.api as sm


In [248]:
df = pd.read_csv("eBayAuctions.csv")

In [249]:
df.columns = df.columns.str.replace(' ', '')

In [250]:
df['Competitive?'] = df['Competitive?'].astype(int)


In [251]:
#a
category_pivot = df.pivot_table(index='Category', values='Competitive?', aggfunc='mean')
print("Mean competitiveness by Category:\n", category_pivot.sort_values('Competitive?'))

currency_pivot = df.pivot_table(index='currency', values='Competitive?', aggfunc='mean')
print("\nMean competitiveness by Currency:\n", currency_pivot)

day_pivot = df.pivot_table(index='endDay', values='Competitive?', aggfunc='mean')
print("\nMean competitiveness by End Day:\n", day_pivot)

duration_pivot = df.pivot_table(index='Duration', values='Competitive?', aggfunc='mean')
print("\nMean competitiveness by Duration:\n", duration_pivot)


Mean competitiveness by Category:
                       Competitive?
Category                          
Health/Beauty             0.171875
EverythingElse            0.235294
Coins/Stamps              0.297297
Pottery/Glass             0.350000
Automotive                0.353933
Jewelry                   0.365854
Books                     0.500000
Clothing/Accessories      0.504202
Toys/Hobbies              0.529915
Antique/Art/Craft         0.564972
Collectibles              0.577406
Music/Movie/Game          0.602978
Home/Garden               0.656863
Computer                  0.666667
Business/Industrial       0.666667
SportingGoods             0.725806
Electronics               0.800000
Photography               0.846154

Mean competitiveness by Currency:
           Competitive?
currency              
EUR           0.551595
GBP           0.687075
US            0.519350

Mean competitiveness by End Day:
         Competitive?
endDay              
Fri         0.466899
Mon         0.67

Several categories show noticeable differences in competitiveness rates. Auctions in categories such as Photography (0.846), Electronics (0.800), and Sporting Goods (0.726) have much higher competitiveness compared to categories like Health/Beauty (0.172), EverythingElse (0.235), and Coins/Stamps (0.297). This suggests that the type of item being sold strongly influences whether an auction becomes competitive.

For currency, auctions listed in GBP (0.687) appear more competitive than those in EUR (0.552) or US (0.519), indicating that currency may also play a role in competitiveness.

There are also differences across ending days. Auctions ending on Monday (0.673) and Thursday (0.604) show higher competitiveness compared to those ending on Saturday (0.427) or Friday (0.467). This suggests that timing of auction closure may influence bidding activity.

Auction duration also shows variation. Five-day auctions (0.687) are the most competitive, while three-day auctions (0.451) are the least competitive. This indicates that duration selection by sellers may affect the likelihood of attracting multiple bids.

Overall, category, duration, and ending day appear to show meaningful differences in competitiveness. To simplify the model while retaining predictive information, categories and days with similar competitiveness rates could be grouped together into broader categories (for example, high, medium, and low competitiveness groups).

In [252]:
#b
categorical_cols = ['Category', 'currency', 'endDay', 'Duration']
X = pd.get_dummies(df[categorical_cols], drop_first=True)

X['sellerRating'] = df['sellerRating']
X['OpenPrice'] = df['OpenPrice']
X['ClosePrice'] = df['ClosePrice']  

y = df['Competitive?']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

logreg_full = LogisticRegression(max_iter=1000)
logreg_full.fit(X_train, y_train)

y_pred_full = logreg_full.predict(X_valid)
accuracy_full = accuracy_score(y_valid, y_pred_full)
cm_full = confusion_matrix(y_valid, y_pred_full)

print("\nFull model accuracy (including closing price):", accuracy_full)
print("Confusion matrix:\n", cm_full)





Full model accuracy (including closing price): 0.779467680608365
Confusion matrix:
 [[286  76]
 [ 98 329]]


C:\Users\jamal\OneDrive\Documents\IDS\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


After splitting the data into 60% training and 40% validation sets, I ran a logistic regression model using all predictors, including closing price. The validation accuracy was 0.7795 (77.95%), meaning the model correctly classified about 77.95% of auctions as competitive versus non-competitive.

The confusion matrix shows that both types of misclassification occurred. There were 76 false positives (predicting competitive when the auction was not competitive) and 98 false negatives (predicting non-competitive when the auction was actually competitive). False negatives were slightly more common than false positives, indicating that the model had more difficulty identifying some competitive auctions correctly.

In [253]:
#c
X_start = X.drop(columns=['ClosePrice'])
X_train_start, X_valid_start, y_train, y_valid = train_test_split(X_start, y, test_size=0.4, random_state=42, stratify=y)

logreg_start = LogisticRegression(max_iter=1000)
logreg_start.fit(X_train_start, y_train)

y_pred_start = logreg_start.predict(X_valid_start)
accuracy_start = accuracy_score(y_valid, y_pred_start)

print("\nAccuracy without closing price:", accuracy_start)



Accuracy without closing price: 0.6501901140684411


C:\Users\jamal\OneDrive\Documents\IDS\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


The accuracy without including closing price is 0.6502 (65.02%), which represents a decrease of 0.1293 (12.93%) compared to the full model accuracy of 77.95%. This indicates that closing price contributes substantial predictive power to the model. However, since closing price is not known at the start of an auction, the reduced model without closing price is more realistic for early prediction despite having lower accuracy.

In [254]:
#d
X_train_sm = sm.add_constant(X_train)  
X_train_sm = X_train_sm.astype(float)
y_train = y_train.astype(int)
logit_model = sm.Logit(y_train, X_train_sm)
result = logit_model.fit()
print("\nLogistic regression summary (full model):")
print(result.summary())

closing_coef = result.params['ClosePrice']
odds_ratio = np.exp(closing_coef)
print(f"\nClose price coefficient: {closing_coef:.4f}, Odds ratio: {odds_ratio:.2f}")


Optimization terminated successfully.
         Current function value: 0.499392
         Iterations 10

Logistic regression summary (full model):
                           Logit Regression Results                           
Dep. Variable:           Competitive?   No. Observations:                 1183
Model:                          Logit   Df Residuals:                     1153
Method:                           MLE   Df Model:                           29
Date:                Sat, 21 Feb 2026   Pseudo R-squ.:                  0.2762
Time:                        14:24:27   Log-Likelihood:                -590.78
converged:                       True   LL-Null:                       -816.17
Covariance Type:            nonrobust   LLR p-value:                 3.472e-77
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const                     

The coefficient for close price is 0.1229, indicating a positive relationship between close price and the probability that an auction is competitive. This means that as the final price increases, the likelihood of the auction being competitive also increases. The odds ratio of 1.13 suggests that for each one-unit increase in close price, the odds of an auction being competitive increase by approximately 13%.

The p-value for close price is 0.000, which is well below the 10% significance level (0.10). Therefore, close price is statistically significant for predicting auction competitiveness.


In [255]:
#e
logreg_l1 = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000)
logreg_l1.fit(X_train_start, y_train)  # exclude closing price

y_pred_l1 = logreg_l1.predict(X_valid_start)
accuracy_l1 = accuracy_score(y_valid, y_pred_l1)

coef_l1 = pd.Series(logreg_l1.coef_[0], index=X_train_start.columns)
selected_predictors = coef_l1[coef_l1 != 0].index.tolist()

print("\nL1-regularized logistic regression accuracy:", accuracy_l1)
print("Selected predictors with L1 penalty:", selected_predictors)


L1-regularized logistic regression accuracy: 0.6565272496831432
Selected predictors with L1 penalty: ['Duration', 'Category_Automotive', 'Category_Books', 'Category_Business/Industrial', 'Category_Coins/Stamps', 'Category_Collectibles', 'Category_Computer', 'Category_Electronics', 'Category_EverythingElse', 'Category_Health/Beauty', 'Category_Home/Garden', 'Category_Jewelry', 'Category_Music/Movie/Game', 'Category_Photography', 'Category_Pottery/Glass', 'Category_SportingGoods', 'Category_Toys/Hobbies', 'currency_GBP', 'endDay_Mon', 'endDay_Sat', 'endDay_Sun', 'endDay_Thu', 'endDay_Tue', 'endDay_Wed', 'sellerRating', 'OpenPrice']


In [256]:
#f
coefficients = pd.Series(logreg_start.coef_[0], index=X_train_start.columns).sort_values(ascending=False)
print("\nTop predictors increasing competitiveness:\n", coefficients.head(10))
print("\nTop predictors decreasing competitiveness:\n", coefficients.tail(10))



Top predictors increasing competitiveness:
 Category_SportingGoods          1.196961
Category_Electronics            1.153926
currency_GBP                    1.019796
endDay_Mon                      0.990609
Category_Photography            0.617690
Category_Business/Industrial    0.326410
Category_Collectibles           0.276003
endDay_Tue                      0.207737
Category_Computer               0.191305
Category_Toys/Hobbies           0.162757
dtype: float64

Top predictors decreasing competitiveness:
 Category_Jewelry          -0.263276
endDay_Sat                -0.285001
Category_Pottery/Glass    -0.314854
endDay_Thu                -0.330866
endDay_Wed                -0.402861
Category_Books            -0.422927
Category_Automotive       -0.545579
Category_EverythingElse   -0.818289
Category_Coins/Stamps     -0.920468
Category_Health/Beauty    -1.382277
dtype: float64


Based on the results, several auction settings appear more likely to lead to a competitive auction. Lower opening prices are associated with a higher probability of competitiveness, likely because they attract more bidders initially. Auctions ending on Monday show higher competitiveness compared to some other days, while auctions ending on Saturday tend to be less competitive. Currency also plays a role, with auctions listed in GBP showing higher competitiveness than others.

Although duration was included in the model, its effect was relatively small compared to factors like opening price and timing. Overall, to increase the likelihood of a competitive auction, sellers should consider setting a lower opening price, choosing favorable ending days such as Monday, and selecting auction settings that encourage early bidding activity.